# Colab GPU Setup
Run these cells in order before executing Node 2, 3, or 4.

In [ ]:
# 1. Mount Drive (large parquets live here)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Extract zipped parquets from Drive → keep on Drive to save Colab disk
import zipfile, os

DRIVE_DIR = '/content/drive/MyDrive/nedbank_raw'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Update file IDs when second zip is ready
zips = {
    'transactions_features.zip': '18VNmRDovoE4lEglCv2PrZETHuUXenzLK',
    'financials_features.zip':   'PASTE_SECOND_FILE_ID_HERE',
}

import gdown
for fname, fid in zips.items():
    out = f'{DRIVE_DIR}/{fname}'
    if not os.path.exists(out.replace('.zip', '.parquet')):
        gdown.download(id=fid, output=out, quiet=False)
        with zipfile.ZipFile(out, 'r') as z:
            z.extractall(DRIVE_DIR)
        os.remove(out)

print(os.listdir(DRIVE_DIR))

In [ ]:
# 3. Upload small local files (Train.csv, Test.csv, demographics_clean.zip)
from google.colab import files
import shutil, zipfile

os.makedirs('data/raw', exist_ok=True)
uploaded = files.upload()

for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('data/raw')
        os.remove(fname)
    else:
        shutil.move(fname, f'data/raw/{fname}')

print(os.listdir('data/raw'))

In [ ]:
# 4. Set env vars (replace with your actual values)
import os
os.environ['HF_TOKEN'] = 'hf_YOUR_TOKEN_HERE'
os.environ['HF_REPO']  = 'Zolisa/nedbank-features'
os.environ['DATA_DIR'] = DRIVE_DIR  # points Node 1 at Drive parquets

In [ ]:
# 5. Install dependencies
!pip install -q polars==1.28.1 huggingface_hub pyarrow python-dotenv lightgbm xgboost==3.2.0 joblib gdown
# TimesFM 2.5 must be installed from source (v2.0 is archived)
!git clone --depth 1 https://github.com/google-research/timesfm.git /tmp/timesfm
!pip install -q -e '/tmp/timesfm[torch]'

In [ ]:
# 6. Clone your repo (after you push to GitHub)
# !git clone https://github.com/YOUR_USERNAME/zindi_nedbank_forecast.git
# %cd zindi_nedbank_forecast